# OCR всего датасета через PP-OCRv5 Mobile

Notebook распознаёт все изображения датасета с нуля на двух T4. Используются `PP-OCRv5_mobile_det` и `eslav_PP-OCRv5_mobile_rec`.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Select Accelerator = GPU T4 x2"
assert torch.cuda.device_count() >= 2, "Need two T4 GPUs"

VENV_ROOT = Path("/kaggle/working/ecup_ppocrv5_mobile_env")
OCR_ENV = VENV_ROOT / "ocr"
OCR_PY = OCR_ENV / "bin" / "python"
OCR_CLI = OCR_ENV / "bin" / "paddleocr"

VENV_ROOT.mkdir(parents=True, exist_ok=True)

def clean_env(cuda=None):
    env = os.environ.copy()
    for key in ["PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "PYTHONUSERBASE"]:
        env.pop(key, None)
    env["PYTHONNOUSERSITE"] = "1"
    env["PYTHONWARNINGS"] = "ignore"
    env["FLAGS_minloglevel"] = "3"
    env["GLOG_minloglevel"] = "3"
    env["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
    if cuda is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(cuda)
    return env

def run_quiet(cmd, env=None, check=True):
    result = subprocess.run(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    if check and result.returncode != 0:
        print("\n".join(result.stdout.splitlines()[-40:]))
        raise RuntimeError(f"Command failed: {cmd}")
    return result

if subprocess.run(
    [sys.executable, "-m", "virtualenv", "--version"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode != 0:
    run_quiet([sys.executable, "-m", "pip", "install", "-q", "-U", "virtualenv"])

healthy = (
    OCR_PY.exists()
    and subprocess.run(
        [str(OCR_PY), "-m", "pip", "--version"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=clean_env(),
    ).returncode == 0
)

if not healthy:
    shutil.rmtree(OCR_ENV, ignore_errors=True)
    run_quiet([
        sys.executable,
        "-m",
        "virtualenv",
        "--no-download",
        "--clear",
        str(OCR_ENV),
    ])

env = clean_env()

run_quiet([OCR_PY, "-m", "pip", "install", "-q", "wrapt"], env=env)
run_quiet(
    [OCR_PY, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"],
    env=env,
)

READY = OCR_ENV / ".ready_ppocrv5_mobile_v1"
if not READY.exists():
    run_quiet([
        OCR_PY,
        "-m",
        "pip",
        "install",
        "-q",
        "paddlepaddle-gpu==3.2.1",
        "-i",
        "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
    ], env=env)
    run_quiet([
        OCR_PY,
        "-m",
        "pip",
        "install",
        "-q",
        "paddleocr==3.4.1",
    ], env=env)
    READY.write_text("ok", encoding="utf-8")

check = run_quiet([
    OCR_PY,
    "-c",
    "import paddle,paddleocr;"
    "print('paddle',paddle.__version__);"
    "print('paddleocr',paddleocr.__version__);"
    "print('cuda',paddle.device.is_compiled_with_cuda());"
    "print('gpus',paddle.device.cuda.device_count())",
], env=clean_env(cuda=0))

print(check.stdout.strip())
assert "cuda True" in check.stdout

HPI_READY = OCR_ENV / ".hpi_gpu_ready"
if not HPI_READY.exists():
    hpi = run_quiet([OCR_CLI, "install_hpi_deps", "gpu"], env=env, check=False)
    if hpi.returncode == 0:
        HPI_READY.write_text("ok", encoding="utf-8")


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working/ecup_ppocrv5_mobile_full")
OUT_DIR = WORK / "ocr_shards"
LOG_DIR = WORK / "logs"
MANIFEST = WORK / "all_image_manifest.csv"

for path in [WORK, OUT_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def norm_id(value):
    try:
        number = float(value)
        if number.is_integer():
            return str(int(number))
    except Exception:
        pass
    return str(value)

def sort_key(path):
    try:
        return 0, int(path.stem)
    except Exception:
        return 1, path.name

def find_data_csv():
    required = {"id", "name", "description", "category", "label"}
    for path in INPUT.rglob("data.csv"):
        try:
            if required <= set(pd.read_csv(path, nrows=3).columns):
                return path
        except Exception:
            pass
    raise FileNotFoundError("E-CUP data.csv not found")

def find_images_root():
    known = [
        Path("/kaggle/input/datasets/fabifvue/ozon2task-images/images"),
        Path("/kaggle/input/ozon2task-images/images"),
        Path("/kaggle/input/ozon2task_images/images"),
    ]
    for path in known:
        if path.exists():
            return path

    candidates = []
    for path in INPUT.rglob("images"):
        if not path.is_dir():
            continue
        try:
            count = sum(1 for item in path.iterdir() if item.is_dir())
        except Exception:
            continue
        if count > 100:
            candidates.append((count, path))

    if not candidates:
        raise FileNotFoundError("images root not found")
    return max(candidates, key=lambda item: item[0])[1]

DATA_CSV = find_data_csv()
IMAGES_ROOT = find_images_root()

df = pd.read_csv(DATA_CSV)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
df["_id"] = df["id"].map(norm_id)

def list_product_images(product_id, normalized_id):
    folder = IMAGES_ROOT / normalized_id
    if not folder.exists():
        return []

    paths = sorted(
        [
            path
            for path in folder.iterdir()
            if path.is_file() and path.suffix.lower() in IMG_EXTS
        ],
        key=sort_key,
    )

    result = []
    for image_idx, path in enumerate(paths):
        try:
            file_bytes = path.stat().st_size
        except Exception:
            file_bytes = 1
        result.append({
            "id": product_id,
            "_id": normalized_id,
            "image_idx": image_idx,
            "image_path": str(path),
            "file_bytes": max(int(file_bytes), 1),
        })
    return result

records = list(df[["id", "_id"]].itertuples(index=False, name=None))
image_rows = []

workers = min(24, max(8, os.cpu_count() or 8))
with ThreadPoolExecutor(max_workers=workers) as executor:
    futures = [
        executor.submit(list_product_images, product_id, normalized_id)
        for product_id, normalized_id in records
    ]
    for future in as_completed(futures):
        image_rows.extend(future.result())

images = pd.DataFrame(image_rows)
assert len(images) > 0, "No images discovered"

order = images.sort_values(
    ["file_bytes", "_id", "image_idx"],
    ascending=[False, True, True],
)

loads = [0, 0]
shard_map = {}

for index, row in order.iterrows():
    shard = int(np.argmin(loads))
    shard_map[index] = shard
    loads[shard] += int(row["file_bytes"])

images["ocr_shard"] = [shard_map[index] for index in images.index]
images.to_csv(MANIFEST, index=False)

print("data:", DATA_CSV)
print("images:", IMAGES_ROOT)
print("products:", len(df))
print("all images:", len(images))
print("GPU0:", int((images["ocr_shard"] == 0).sum()))
print("GPU1:", int((images["ocr_shard"] == 1).sum()))
print("manifest:", MANIFEST)


In [ ]:
WORKER = WORK / "ppocrv5_mobile_worker.py"

WORKER_CODE = r'''
import argparse
import csv
import json
import os
import re
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["FLAGS_minloglevel"] = "3"
os.environ["GLOG_minloglevel"] = "3"
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

import numpy as np
import pandas as pd


def norm_id(x):
    try:
        f = float(x)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass
    return str(x)


def clean_text(s):
    s = "" if s is None else str(s)
    s = s.replace("\x00", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def result_dict(res):
    obj = getattr(res, "json", None)
    if callable(obj):
        obj = obj()
    if obj is None:
        try:
            obj = dict(res)
        except Exception:
            return {}
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except Exception:
            return {}
    if not isinstance(obj, dict):
        return {}
    if isinstance(obj.get("res"), dict):
        obj = obj["res"]
    if isinstance(obj.get("prunedResult"), dict):
        obj = obj["prunedResult"]
    return obj


def extract_ocr(res):
    obj = result_dict(res)
    texts = obj.get("rec_texts", [])
    scores = obj.get("rec_scores", [])

    try:
        texts = list(texts)
    except Exception:
        texts = []

    try:
        scores = np.asarray(scores, dtype=float).reshape(-1).tolist()
    except Exception:
        scores = []

    kept_texts = []
    kept_scores = []

    for i, txt in enumerate(texts):
        txt = clean_text(txt)
        if not txt:
            continue
        kept_texts.append(txt)
        if i < len(scores) and np.isfinite(scores[i]):
            kept_scores.append(float(scores[i]))

    text = clean_text("\n".join(kept_texts))
    mean_score = float(np.mean(kept_scores)) if kept_scores else np.nan
    min_score = float(np.min(kept_scores)) if kept_scores else np.nan
    return text, mean_score, min_score, len(kept_texts)


def build_pipeline(mode):
    import paddle
    from paddleocr import PaddleOCR

    assert paddle.device.is_compiled_with_cuda()
    assert paddle.device.cuda.device_count() >= 1

    common = dict(
        text_detection_model_name="PP-OCRv5_mobile_det",
        text_recognition_model_name="eslav_PP-OCRv5_mobile_rec",
        text_recognition_batch_size=64,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        text_det_limit_side_len=960,
        text_det_limit_type="max",
        text_rec_score_thresh=0.0,
        device="gpu:0",
    )

    if mode == "hpi":
        return PaddleOCR(
            enable_hpi=True,
            precision="fp16",
            use_tensorrt=False,
            **common,
        )

    return PaddleOCR(
        enable_hpi=False,
        **common,
    )


def existing_pairs(out):
    if not out.exists() or out.stat().st_size == 0:
        return set()
    try:
        old = pd.read_csv(out, usecols=["_id", "image_idx"])
        old["_id"] = old["_id"].map(norm_id)
        old["image_idx"] = old["image_idx"].astype(int)
        return set(zip(old["_id"], old["image_idx"]))
    except Exception:
        return set()


def write_row(writer, handle, row, fsync=False):
    writer.writerow(row)
    handle.flush()
    if fsync:
        os.fsync(handle.fileno())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--shard", type=int, required=True)
    ap.add_argument("--output", required=True)
    ap.add_argument("--mode", choices=["hpi", "standard"], default="standard")
    ap.add_argument("--chunk", type=int, default=64)
    ap.add_argument("--limit", type=int, default=0)
    args = ap.parse_args()

    manifest = pd.read_csv(args.manifest)
    manifest["_id"] = manifest["_id"].map(norm_id)
    manifest["image_idx"] = manifest["image_idx"].astype(int)

    part = manifest[manifest["ocr_shard"] == args.shard].copy().reset_index(drop=True)

    out = Path(args.output)
    out.parent.mkdir(parents=True, exist_ok=True)

    done = existing_pairs(out)
    keep = [
        (rid, int(idx)) not in done
        for rid, idx in zip(part["_id"], part["image_idx"])
    ]
    pending = part[keep].copy().reset_index(drop=True)

    if args.limit > 0:
        pending = pending.head(args.limit).copy().reset_index(drop=True)

    print(
        f"START shard={args.shard} mode={args.mode} total={len(part)} "
        f"already_in_output={len(part)-len(pending)} pending={len(pending)}",
        flush=True,
    )

    if len(pending) == 0:
        print(f"DONE shard={args.shard} rows=0", flush=True)
        return

    t_init = time.time()
    pipeline = build_pipeline(args.mode)
    print(
        f"PIPELINE_READY shard={args.shard} mode={args.mode} "
        f"seconds={time.time()-t_init:.1f}",
        flush=True,
    )

    columns = [
        "id",
        "_id",
        "image_idx",
        "image_path",
        "ocr_text_image",
        "ocr_char_count_image",
        "ocr_seconds_image_batch_avg",
        "ocr_batch_used",
        "ocr_shard",
        "ocr_backend",
        "ocr_mean_score",
        "ocr_min_score",
        "ocr_text_boxes",
    ]

    need_header = not out.exists() or out.stat().st_size == 0
    handle = open(out, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(handle, fieldnames=columns)

    if need_header:
        writer.writeheader()
        handle.flush()

    pos = 0
    saved_since_fsync = 0
    started = time.time()

    while pos < len(pending):
        cur = min(args.chunk, len(pending) - pos)
        batch = pending.iloc[pos:pos+cur].copy().reset_index(drop=True)
        paths = batch["image_path"].astype(str).tolist()
        batch_started = time.time()

        try:
            generator = pipeline.predict_iter(
                paths,
                use_doc_orientation_classify=False,
                use_doc_unwarping=False,
                use_textline_orientation=False,
                text_det_limit_side_len=960,
                text_det_limit_type="max",
                text_rec_score_thresh=0.0,
            )

            produced = 0
            prev = time.time()

            for j, res in enumerate(generator):
                if j >= len(batch):
                    raise RuntimeError("OCR returned more results than inputs")

                row = batch.iloc[j]
                now = time.time()
                sec = now - prev
                prev = now

                text, mean_score, min_score, n_boxes = extract_ocr(res)

                write_row(
                    writer,
                    handle,
                    {
                        "id": row["id"],
                        "_id": row["_id"],
                        "image_idx": int(row["image_idx"]),
                        "image_path": row["image_path"],
                        "ocr_text_image": text,
                        "ocr_char_count_image": len(text),
                        "ocr_seconds_image_batch_avg": sec,
                        "ocr_batch_used": int(len(batch)),
                        "ocr_shard": int(args.shard),
                        "ocr_backend": "PP-OCRv5_mobile_det+eslav_PP-OCRv5_mobile_rec",
                        "ocr_mean_score": mean_score,
                        "ocr_min_score": min_score,
                        "ocr_text_boxes": int(n_boxes),
                    },
                    fsync=False,
                )
                produced += 1
                saved_since_fsync += 1

                if saved_since_fsync >= 32:
                    os.fsync(handle.fileno())
                    saved_since_fsync = 0

            if produced != len(batch):
                raise RuntimeError(
                    f"OCR result count mismatch: produced={produced}, expected={len(batch)}"
                )

            pos += len(batch)

        except Exception as batch_error:
            print(
                f"BATCH_FALLBACK shard={args.shard} pos={pos} size={len(batch)} "
                f"error={type(batch_error).__name__}",
                flush=True,
            )

            for _, row in batch.iterrows():
                t0 = time.time()
                text = ""
                mean_score = np.nan
                min_score = np.nan
                n_boxes = 0
                try:
                    results = list(
                        pipeline.predict_iter(
                            [str(row["image_path"])],
                            use_doc_orientation_classify=False,
                            use_doc_unwarping=False,
                            use_textline_orientation=False,
                            text_det_limit_side_len=960,
                            text_det_limit_type="max",
                            text_rec_score_thresh=0.0,
                        )
                    )
                    if results:
                        text, mean_score, min_score, n_boxes = extract_ocr(results[0])
                except Exception:
                    pass

                write_row(
                    writer,
                    handle,
                    {
                        "id": row["id"],
                        "_id": row["_id"],
                        "image_idx": int(row["image_idx"]),
                        "image_path": row["image_path"],
                        "ocr_text_image": text,
                        "ocr_char_count_image": len(text),
                        "ocr_seconds_image_batch_avg": time.time() - t0,
                        "ocr_batch_used": 1,
                        "ocr_shard": int(args.shard),
                        "ocr_backend": "PP-OCRv5_mobile_det+eslav_PP-OCRv5_mobile_rec",
                        "ocr_mean_score": mean_score,
                        "ocr_min_score": min_score,
                        "ocr_text_boxes": int(n_boxes),
                    },
                    fsync=False,
                )
                saved_since_fsync += 1

                if saved_since_fsync >= 32:
                    os.fsync(handle.fileno())
                    saved_since_fsync = 0

            pos += len(batch)

        elapsed = time.time() - started
        rate = pos / max(elapsed, 1e-6)
        eta = (len(pending) - pos) / max(rate, 1e-6) / 60

        print(
            f"PROGRESS shard={args.shard} {pos}/{len(pending)} "
            f"rate={rate:.3f} image/s chunk={cur} ETA={eta:.1f}m",
            flush=True,
        )

    handle.flush()
    os.fsync(handle.fileno())
    handle.close()

    print(
        f"DONE shard={args.shard} processed={pos} "
        f"seconds={time.time()-started:.1f}",
        flush=True,
    )


if __name__ == "__main__":
    main()
'''

WORKER.write_text(WORKER_CODE, encoding="utf-8")
compile(WORKER_CODE, str(WORKER), "exec")
print(WORKER)


In [ ]:
import time
from IPython.display import clear_output

def worker_env(gpu):
    env = clean_env(cuda=gpu)
    env["FLAGS_allocator_strategy"] = "auto_growth"
    env["FLAGS_fraction_of_gpu_memory_to_use"] = "0.90"
    return env

def tail(path, n=12):
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
    useful = [
        line
        for line in lines
        if line.startswith(("START ", "PIPELINE_READY ", "PROGRESS ", "DONE ", "BATCH_FALLBACK "))
    ]
    return "\n".join((useful if useful else lines)[-n:])

SMOKE_OUT = WORK / "smoke.csv"
SMOKE_LOG = LOG_DIR / "smoke.log"
modes = ["hpi", "standard"] if HPI_READY.exists() else ["standard"]
RUN_MODE = None

for mode in modes:
    SMOKE_OUT.unlink(missing_ok=True)
    cmd = [
        str(OCR_PY),
        "-u",
        str(WORKER),
        "--manifest",
        str(MANIFEST),
        "--shard",
        "0",
        "--output",
        str(SMOKE_OUT),
        "--mode",
        mode,
        "--chunk",
        "8",
        "--limit",
        "8",
    ]

    with SMOKE_LOG.open("w", encoding="utf-8") as handle:
        result = subprocess.run(
            cmd,
            stdout=handle,
            stderr=subprocess.STDOUT,
            env=worker_env(0),
        )

    if result.returncode == 0 and SMOKE_OUT.exists() and len(pd.read_csv(SMOKE_OUT)) > 0:
        RUN_MODE = mode
        break

assert RUN_MODE is not None, SMOKE_LOG.read_text(encoding="utf-8", errors="ignore")

processes = []
logs = []

for shard in [0, 1]:
    output = OUT_DIR / f"ocr_mobile_gpu{shard}.csv"
    log = LOG_DIR / f"ocr_mobile_gpu{shard}.log"
    cmd = [
        str(OCR_PY),
        "-u",
        str(WORKER),
        "--manifest",
        str(MANIFEST),
        "--shard",
        str(shard),
        "--output",
        str(output),
        "--mode",
        RUN_MODE,
        "--chunk",
        "64",
    ]

    handle = log.open("w", encoding="utf-8")
    process = subprocess.Popen(
        cmd,
        stdout=handle,
        stderr=subprocess.STDOUT,
        env=worker_env(shard),
    )
    processes.append((process, handle))
    logs.append(log)

while any(process.poll() is None for process, _ in processes):
    time.sleep(15)
    clear_output(wait=True)
    print(f"PP-OCRv5 Mobile | mode={RUN_MODE}")
    for shard, ((process, _), log) in enumerate(zip(processes, logs)):
        print(f"\nGPU{shard} rc={process.poll()}")
        print(tail(log) or "starting...")

for process, handle in processes:
    rc = process.wait()
    handle.close()
    if rc != 0:
        raise RuntimeError(f"OCR worker failed with code {rc}")

print("OCR complete")


In [ ]:
import re
import zipfile
from IPython.display import FileLink, display

parts = []
for shard in [0, 1]:
    path = OUT_DIR / f"ocr_mobile_gpu{shard}.csv"
    part = pd.read_csv(path)
    part["_id"] = part["_id"].map(norm_id)
    part["image_idx"] = part["image_idx"].astype(int)
    parts.append(part)

ocr_images = pd.concat(parts, ignore_index=True)
ocr_images = (
    ocr_images
    .drop_duplicates(["_id", "image_idx"], keep="last")
    .sort_values(["_id", "image_idx"])
    .reset_index(drop=True)
)

if len(ocr_images) != len(images):
    expected = set(zip(images["_id"], images["image_idx"]))
    actual = set(zip(ocr_images["_id"], ocr_images["image_idx"]))
    missing = expected - actual
    raise RuntimeError(f"Missing OCR rows: {len(missing)}")

def normalize_block(value):
    text = "" if pd.isna(value) else str(value)
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

product_rows = []

for product_id, group in ocr_images.groupby("_id", sort=False):
    group = group.sort_values("image_idx")
    blocks = []

    for _, row in group.iterrows():
        text = normalize_block(row["ocr_text_image"])
        if text:
            blocks.append(f"[Фото {int(row['image_idx']) + 1}]\n{text}")

    text = "\n\n".join(blocks)
    product_rows.append({
        "_id": product_id,
        "ocr_text": text,
        "ocr_images_done": int(len(group)),
        "ocr_images_nonempty": int(
            group["ocr_text_image"].fillna("").astype(str).str.len().gt(0).sum()
        ),
        "ocr_char_count": len(text),
    })

ocr_products = pd.DataFrame(product_rows)
product_base = df[["id", "_id", "name", "description", "category", "label"]].copy()

ocr_products = product_base.merge(
    ocr_products,
    on="_id",
    how="left",
    validate="one_to_one",
)
ocr_products["ocr_text"] = ocr_products["ocr_text"].fillna("")
ocr_products["ocr_images_done"] = ocr_products["ocr_images_done"].fillna(0).astype(int)
ocr_products["ocr_images_nonempty"] = ocr_products["ocr_images_nonempty"].fillna(0).astype(int)
ocr_products["ocr_char_count"] = ocr_products["ocr_char_count"].fillna(0).astype(int)

IMAGE_OUT = WORK / "ocr_all_images_high_quality.csv"
PRODUCT_OUT = WORK / "ocr_all_products_high_quality.csv"

ocr_images.to_csv(IMAGE_OUT, index=False)
ocr_products.to_csv(PRODUCT_OUT, index=False)

ZIP_PATH = WORK / "ocr_all_dataset_ppocrv5_mobile.zip"
ZIP_PATH.unlink(missing_ok=True)

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    archive.write(IMAGE_OUT, IMAGE_OUT.name)
    archive.write(PRODUCT_OUT, PRODUCT_OUT.name)

print("images:", len(ocr_images))
print("products:", len(ocr_products))
print("products with OCR:", int(ocr_products["ocr_text"].str.len().gt(0).sum()))
print(IMAGE_OUT)
print(PRODUCT_OUT)

display(FileLink(str(ZIP_PATH)))
